### now I try to construct a full work flow to select, extract, generate psf and transform kernel

### load catalog of all fields

apply a selection rule of sn_Ha(b) > 5 

In [6]:
import numpy as np 
from astropy.io import fits
import matplotlib.pyplot as plt
from astropy.table import Table
from pathlib import Path
from photutils.psf.matching import *
from astropy.convolution import convolve_fft
import webbpsf
from astropy.coordinates import Angle
from astropy import units as u
import matplotlib.colors as colors
import os
from tqdm import tqdm
import time

import gc
import warnings
from astropy.io.fits.verify import VerifyWarning 
import matplotlib.cbook
warnings.simplefilter('ignore', VerifyWarning)
warnings.simplefilter('ignore', UserWarning) 

In [7]:
def cat_select():
    #load catalog
    data = Table.read('spectra-fitting.fits')

    print('num objs before selection',len(data))
    data =  data[
        np.logical_and(data['sn_Ha'] >= 5, data['sn_Hb'] >= 5) #select sn>=5
                ]
    print('num objs after selection',len(data))

    #save catalog after selection
    data[
        'field','id','redshift','ra','dec','flux_Ha','err_Ha','sn_Ha','flux_Hb','err_Hb','sn_Hb'
        ].write('spectra-fitting_selected.fits',overwrite=True)

    return Table.read('spectra-fitting_selected.fits')
obj_lis = cat_select()

num objs before selection 131305
num objs after selection 2044


### download spectrum from server.

It is recommended that to run the script downloadSpectra in prompt because of potential stability issue

In [8]:
#import subprocess

# Define the command to run
command = [
    'C:/ba project/.conda/python.exe', 
    'downloadSpectra.py', 
    'spectra-fitting_selected.fits', 
    '--ncpu', '15'  
]

"""
try:
    # Run the command
    #subprocess.run(command, check=True)
except subprocess.CalledProcessError as e:
    print(f'An error occurred: {e}')
"""

"\ntry:\n    # Run the command\n    #subprocess.run(command, check=True)\nexcept subprocess.CalledProcessError as e:\n    print(f'An error occurred: {e}')\n"

### extract ha hb line maps from full.fits data

In [ ]:
#define a handy little function to return the desired file path
def file_path(obj,prefix,filetype='fits'):
    if   filetype == 'fits':
        return f"data\\{obj['field']}\\{obj['field']}_{str(obj['id']).zfill(5)}.{prefix}.{filetype}"
    elif filetype == 'png':
        return f"png\\{obj['field']}\\{obj['field']}_{str(obj['id']).zfill(5)}.{prefix}.{filetype}"
   
    

"""
pass obj from obj_lis to extract ha hb lines
return: HDUlist with the following entry:
0 primary extension, same as original file
1 line-fit results
2 segmentation map
3 clear filter maps
4,5 Ha line map & line weight
6,7 Hb line map & line weight

"""
def exrtract_HaHb(hdu):


    #set up a crop of 50x50 pix in the center
    center_size = 50; shape = hdu[5].shape[0]
    #start index: si and end index: ei
    si = (shape - center_size) // 2; 
    ei = si + center_size


    new_file = fits.HDUList()
    #save primary extension
    new_file.append(hdu[0])
    #save line-fit info
    new_file.append(hdu[1])
    """
    select segmentation map[4]
    also save 1 DSCI image for comparison [5]
    """
    
    for i in [4,5]: 
        hdu[i].data = hdu[i].data[si:ei,si:ei]
        new_file.append(hdu[i])

    #loop to select ha hb line maps
    for image in hdu:
        if image.ver in ['Ha','Hb'] and (image.name == 'LINE' or image.name == 'LINEWHT'):
            image.data = image.data[si:ei,si:ei]
            image.name = f'{image.name}_{image.ver}'
            new_file.append(image)

    return new_file


### calculate monochromatic PSFs for Ha Hb line maps, generating kernels for Hb psf matching

In [10]:
# filter_data
NIRISS_filters = [
    {"name": "F090W", "lambda_min": 0.796, "lambda_max": 1.005},
    {"name": "F115W", "lambda_min": 1.013, "lambda_max": 1.283},
    {"name": "F150W", "lambda_min": 1.330, "lambda_max": 1.671},
    {"name": "F200W", "lambda_min": 1.751, "lambda_max": 2.226},
    {"name": "F277W", "lambda_min": 2.413, "lambda_max": 3.143},
    {"name": "F356W", "lambda_min": 3.140, "lambda_max": 4.068},
    {"name": "F444W", "lambda_min": 3.880, "lambda_max": 5.023}
]

#calculate mono psf
def mono_webbpsf(wavelength,filter):
    niriss = webbpsf.NIRISS()
    niriss.filter = filter
    return niriss.calc_psf(fov_pixels=50,monochromatic=wavelength)[3]

def calc_psf(z):
    #use redshift to get line position
    waves = [6.563e-7 *(1+z), 4.861e-7 *(1+z)]
    filters = []
    for wave in waves:
        for f in NIRISS_filters:
            if f["lambda_min"] <= wave*1e6 <= f["lambda_max"]:
                filters.append( f"{f['name']}")
                continue
    if len(filters) == 2:
        return mono_webbpsf(waves[0],filters[0]), mono_webbpsf(waves[1],filters[1])
    else:
        return [0,0]

#generate kernel for convolving hb for psf matching
def gen_kernel(psf_hb,psf_ha):
     #generate kernel to match psf_Hb to psf_Ha
        kernel = fits.ImageHDU()
        window = CosineBellWindow(alpha=1.2)
        kernel.data = create_matching_kernel(psf_hb.data,psf_ha.data,window=window)
        kernel.name = 'PSF_MATCH'
        return kernel


def psf_convolve(hb,kernel):
    return fits.ImageHDU(
         data = convolve_fft(hb.data,kernel.data),
         header = hb.header,
         name = 'LINE_HB_CONVOLVED')




### define single plot function

In [11]:
def single_plot(disp,ra,dec,pix,title,ax,style='gray'):

    norm = colors.Normalize(vmin=0)
    if 'BALMER_DECREM' in title:
        norm = colors.Normalize(vmin=-5,vmax=5)
    im    =  ax.imshow(disp
                    ,norm = norm
                    ,origin='lower', cmap = style)
    fig.colorbar(im,ax=ax)
    
    
    plt.title(f'{title}, pixel scale: {pix}\'')
    # Set axis labels
    ax.set_xlabel(f"Ra  +  {Angle(round(ra,4),unit='deg').to_string(unit=u.degree, sep=('h', 'm', 's'))}")
    ax.set_ylabel(f"Dec + {Angle(round(dec,4),unit='deg').to_string(unit=u.degree, sep=('deg', 'm', 's'))}")
    
    # Add N and E directions with arrows
    ax.text(0.9, 0.9, 'N', transform=ax.transAxes, ha='center', va='center', color='white', fontsize=14)
    ax.text(0.1, 0.1, 'E', transform=ax.transAxes, ha='center', va='center', color='white', fontsize=14)

### process datasets

In [13]:
obj_lis = Table.read('spectra-fitting_selected.fits')
row_to_remove = []
step = 0
for index in tqdm(range(len(obj_lis)),
                  desc="Processing Table",
                  ncols=100):

    obj=obj_lis[index]
    if os.path.exists(file_path(obj=obj,prefix='extract')):
        continue

    path = file_path(obj,'full')
    with fits.open(path) as hdu:
    
        #extract HaHb from full data
        extracted = exrtract_HaHb(hdu)
        
        #calculate monochromatic psf
        #only proceed if psf is correctly calculated           
        psf_ha,psf_hb =calc_psf(extracted[0].header['redshift'])
        if psf_ha != 0 and psf_hb !=0:
            psf_ha.name = 'PSF_HA'
            psf_hb.name = 'PSF_HB'

            #generate kernel and convolve
            kernel = gen_kernel(psf_hb,psf_ha)
            hb_convolve =  psf_convolve(extracted[6],kernel)


            #balmer decrement
            #note this process can be improved by removing Ha[NII] mixing from 
            balmer = fits.ImageHDU(data=extracted[4].data/hb_convolve.data, name = 'BALMER_DECREM')
            
            
            #savefile
            extracted.append(psf_ha)
            extracted.append(psf_hb)
            extracted.append(kernel)
            extracted.append(hb_convolve) 
            extracted.append(balmer)
            extracted.writeto(file_path(obj=obj,prefix='extract'),overwrite=True)
        else:
            row_to_remove.append(index)   

        step += 1
        if step >= 10:
            gc.collect()   
            step=0  

print('number of objects before psf matching',len(obj_lis))
obj_lis.remove_rows(row_to_remove)
print('number of objects after  psf matching',len(obj_lis))
obj_lis.write('spectra-fitting_selected_psfmatched.fits',overwrite=True)




Processing Table:   0%|                                                    | 0/2044 [00:00<?, ?it/s]

Processing Table: 100%|████████████████████████████████████████| 2044/2044 [00:15<00:00, 133.83it/s]

number of objects before psf matching 2044
number of objects after  psf matching 1917


### plot

In [14]:
obj_lis = Table.read('spectra-fitting_selected_psfmatched.fits')
step = 0

for index in tqdm(range(len(obj_lis)),
                  desc="Processing Table",
                  ncols=100):

    obj=obj_lis[index]


    if os.path.exists(file_path(obj=obj,prefix='fullplot',filetype='png')):
        continue
    
    path = file_path(obj=obj,prefix='extract')
    with fits.open(path) as extracted:
        #mask = extracted[2].data 
        #mask[mask==extracted[0].header['id']] = 1
        #mask[mask!=1] = 0

# this the part for plot the calculated result
        ra    =  obj['ra']
        dec   =  obj['dec']
        pix   =  extracted[3].header['PIXASEC'] #arcsec

        fig = plt.figure(figsize=(15,15))   
        stuff_to_plot = [3,10,4,6,11,12]
        for index in range(len(stuff_to_plot)):
            #parameter control
            i = stuff_to_plot[index]    
            disp  =  extracted[i].data
            title =  obj['field'],obj['id'],extracted[i].name
            ax    =  fig.add_subplot(int(f'32{index+1}')) 
            single_plot(disp,ra,dec,pix,title,ax,style='binary_r')
        plt.savefig(file_path(obj=obj,prefix='fullplot',filetype='png'))
        plt.close('all')
        step += 1
        if step >= 10:
            gc.collect()   
            step=0  


Processing Table: 100%|███████████████████████████████████████| 1917/1917 [00:00<00:00, 2940.98it/s]
